# 01. Data Cleaning & Feature Engineering

**Th?nh vi?n ph? tr?ch:** Th?nh vi?n 1 - Data Engineering & Preprocessing

**M?c ti?u:**
1. L?c d? li?u m?a gi?i NBA t? n?m 1980 tr? ?i (`load_and_filter_seasons`).
2. X? l? c?c tr??ng h?p c?u th? ??i ??i gi?a m?a (`deduplicate_traded_players`) - ?u ti?n gi? d?ng t?ng h?p `TOT`/`2TM`/`3TM`.
3. Chu?n h?a Z-score theo t?ng m?a gi?i (`add_season_zscores`) ?? kh? kh?c bi?t th?i k?.
4. Ph?n chia t?p d? li?u theo th? t? th?i gian (`split_by_season`): Train (<= 2016), Validation (2017-2019), Test (>= 2020).
5. X?y d?ng scikit-learn preprocessing pipeline (`build_preprocessing_pipeline`).
6. Xu?t c?c file d? li?u ?? x? l? v?o th? m?c `data/processed/`.

In [ ]:
import sys
from pathlib import Path

# Th?m th? m?c g?c d? ?n v?o sys.path ?? import t? src
ROOT_DIR = Path.cwd().resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import numpy as np
import pandas as pd
from src.preprocessing import (
    load_and_filter_seasons,
    deduplicate_traded_players,
    add_season_zscores,
    split_by_season,
    build_preprocessing_pipeline
)

print("Imports th?nh c?ng t? src.preprocessing!")

## B??c 1: T?i d? li?u th? & L?c m?a gi?i t? n?m 1980 (`load_and_filter_seasons`)
L?c b? c?c m?a gi?i tr??c 1980 (k? nguy?n tr??c v?ch n?m 3 ?i?m) ?? chu?n h?a ??c tr?ng chi?n thu?t.

In [ ]:
raw_data_path = ROOT_DIR / "data" / "raw" / "nba_player_stats.csv"

# T?i d? li?u th?c t? ho?c kh?i t?o d? li?u m?u n?u file ch?a t?i v?
try:
    df_raw = pd.read_csv(raw_data_path)
    print(f"?? t?i {len(df_raw)} d?ng t? {raw_data_path}")
except FileNotFoundError:
    print(f"Ch?a c? file th?c t? t?i {raw_data_path}. T?o d? li?u m?u ph?c v? ki?m th? pipeline:")
    np.random.seed(42)
    demo_players = ["LeBron James", "Stephen Curry", "Giannis Antetokounmpo", "James Harden", "Kevin Durant"]
    demo_seasons = [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
    rows = []
    for s in demo_seasons:
        for p in demo_players:
            rows.append({
                "player": p,
                "season": s,
                "team": "TOT" if (p == "James Harden" and s == 2021) else "LAL",
                "pts": np.random.uniform(18, 32),
                "reb": np.random.uniform(4, 12),
                "ast": np.random.uniform(3, 10),
                "mp": np.random.uniform(28, 38),
                "is_all_star": 1 if np.random.rand() > 0.3 else 0
            })
    # Th?m d?ng trade ?? ki?m th? deduplication
    rows.append({"player": "James Harden", "season": 2021, "team": "HOU", "pts": 24.8, "reb": 5.1, "ast": 10.4, "mp": 36.3, "is_all_star": 1})
    rows.append({"player": "James Harden", "season": 2021, "team": "BKN", "pts": 24.6, "reb": 8.5, "ast": 10.9, "mp": 36.6, "is_all_star": 1})
    df_raw = pd.DataFrame(rows)

# L?c m?a gi?i >= 1980
df_filtered = load_and_filter_seasons(df_raw, min_season=1980)
print(f"S? d?ng sau khi l?c m?a gi?i >= 1980: {len(df_filtered)}")
df_filtered.head()

## B??c 2: X? l? c?u th? ??i ??i gi?a m?a (`deduplicate_traded_players`)
?u ti?n gi? d?ng t?ng h?p `TOT`, `2TM`, `3TM` ?? m?i c?p `(player, season)` c? ??ng 1 d?ng ??i di?n duy nh?t.

In [ ]:
df_dedup = deduplicate_traded_players(df_filtered, player_col="player", season_col="season", team_col="team")
print(f"S? d?ng tr??c deduplication: {len(df_filtered)}, sau: {len(df_dedup)}")

# Ki?m tra xem c?n c?p tr?ng kh?ng
n_dups = df_dedup.duplicated(subset=["player", "season"]).sum()
print(f"S? c?p tr?ng (player, season): {n_dups}")
df_dedup[df_dedup["player"] == "James Harden"]

## B??c 3: Chu?n h?a Z-score theo t?ng m?a gi?i (`add_season_zscores`)
Kh? s? kh?c bi?t th?i k? (Pace v? League Average qua t?ng n?m) b?ng c?ch t?nh Z-score trong n?i b? m?i m?a gi?i:
$$Z = \frac{X - \mu_{season}}{\sigma_{season}}$$

In [ ]:
stat_cols = [c for c in ["pts", "reb", "ast", "mp"] if c in df_dedup.columns]
df_zscored = add_season_zscores(df_dedup, feature_cols=stat_cols, season_col="season", suffix="_zscore")

zscore_cols = [f"{c}_zscore" for c in stat_cols]
print("C?c c?t Z-score m?i:", zscore_cols)
df_zscored[["player", "season"] + zscore_cols].head()

## B??c 4: Chia t?p d? li?u theo th? t? th?i gian (`split_by_season`)
- Train: <= 2016
- Validation: 2017 - 2019
- Test: >= 2020

In [ ]:
train_df, val_df, test_df = split_by_season(df_zscored, train_end=2016, val_end=2019, season_col="season")
print(f"Train set: {len(train_df)} d?ng (M?a {train_df['season'].min()} - {train_df['season'].max()})")
print(f"Val set:   {len(val_df)} d?ng (M?a {val_df['season'].min()} - {val_df['season'].max()})")
print(f"Test set:  {len(test_df)} d?ng (M?a {test_df['season'].min()} - {test_df['season'].max()})")

## B??c 5: Scikit-learn Preprocessing Pipeline (`build_preprocessing_pipeline`)
X?y d?ng pipeline g?m `SimpleImputer(strategy='median')` v? `StandardScaler()`. Pipeline ch? fit tr?n t?p Train.

In [ ]:
pipeline = build_preprocessing_pipeline(numeric_features=zscore_cols)

# Fit ch? tr?n t?p Train v? bi?n ??i c?c t?p
X_train_trans = pipeline.fit_transform(train_df)
X_val_trans = pipeline.transform(val_df)
X_test_trans = pipeline.transform(test_df)

print(f"K?ch th??c X_train sau pipeline: {X_train_trans.shape}")
print(f"K?ch th??c X_val sau pipeline:   {X_val_trans.shape}")
print(f"K?ch th??c X_test sau pipeline:  {X_test_trans.shape}")

## B??c 6: L?u d? li?u ?? x? l? v?o `data/processed/`

In [ ]:
processed_dir = ROOT_DIR / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(processed_dir / "train.csv", index=False)
val_df.to_csv(processed_dir / "val.csv", index=False)
test_df.to_csv(processed_dir / "test.csv", index=False)
print(f"?? l?u th?nh c?ng train.csv, val.csv, test.csv v?o {processed_dir}")